# 03 — Pretraitement

Normalise les champs textuels des quatre sources, puis ecrit une version
**partitionnee par departement** des tables SIRENE.

Cette partition est le point cle de la performance des phases 2 et 3. Le
blocking etant departemental, chaque phase lit un departement a la fois au lieu
de charger les 15 millions d'unites legales : sur le departement 75, mesure a
1,23 million de lignes, la lecture passe de 35 secondes et 15 Go a moins de 9
secondes et 1,25 Go.

Les tables SIRENE sont traitees par lots pour ne jamais tout charger en memoire.

In [1]:
import sys, os, shutil
sys.path.insert(0, os.path.abspath('..'))
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from src.pretraitement import pretraiter_pm, pretraiter_ege, pretraiter_etab, pretraiter_ul
from src import cog
from src.display import afficher_tableau
from config.settings import (
    FINESS_PM_RAW, FINESS_EGE_RAW, SIRENE_ETAB_RAW, SIRENE_UL_RAW,
    FINESS_PM_CLEAN, FINESS_EGE_CLEAN, SIRENE_ETAB_CLEAN, SIRENE_UL_CLEAN,
    SIRENE_ETAB_PARTITIONNE, SIRENE_UL_PARTITIONNE,
    PROCESSED_DIR, TAILLE_CHUNK_PARQUET,
    COG_TABLE_PASSAGE, RESULTS_DIR,
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Personnes morales

In [2]:
df_pm = pretraiter_pm(pd.read_parquet(FINESS_PM_RAW))
df_pm.to_parquet(FINESS_PM_CLEAN, index=False)
print(f"PM : {len(df_pm):,} -> {FINESS_PM_CLEAN}")

afficher_tableau(
    df_pm[["NB_PmSmsseId", "finess_norm_pm", "siren_norm_pm", "raisonsociale_norm_pm",
           "libelle_voie_complet_pm", "cdcommune_norm_pm", "dept_pm"]],
    "Apercu PM pretraitees")

PM : 56,193 -> /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/processed/finess_pm_clean.parquet


NB_PmSmsseId,finess_norm_pm,siren_norm_pm,raisonsociale_norm_pm,libelle_voie_complet_pm,cdcommune_norm_pm,dept_pm
2,010000156,772201489,CLINIQUE CONVERT,AVENUE JASSERON,01053,01
6,010000222,546120122,CLINIQUE SOUFFLE PONTET,RUE CHAPELLE,01185,01
9,010000255,779309194,ASS ACCUEIL FORMA INSERT PERSON SOURD,RUE LYCEE,01053,01
17,010000339,260100011,RESIDENCE FONTELUNE,RUE COMMUNE 1871,01004,01
18,010000347,260100029,RESIDENCE URFE,RUE ECOLE PRIVEE,01159,01


## 2. Etablissements

In [3]:
df_ege = pretraiter_ege(pd.read_parquet(FINESS_EGE_RAW))
df_ege.to_parquet(FINESS_EGE_CLEAN, index=False)
print(f"EGE : {len(df_ege):,} -> {FINESS_EGE_CLEAN}")

afficher_tableau(
    df_ege[["NB_EgeId", "NB_PmSmsseId", "finess_norm_ege", "siret_norm_ege",
            "raisonsociale_norm_ege", "libelle_voie_complet_ege",
            "cdcommune_norm_ege", "dept_ege"]],
    "Apercu EGE pretraites")

EGE : 104,649 -> /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/processed/finess_ege_clean.parquet


NB_EgeId,NB_PmSmsseId,finess_norm_ege,siret_norm_ege,raisonsociale_norm_ege,libelle_voie_complet_ege,cdcommune_norm_ege,dept_ege
1,580,010000024,26010004500012,CENTRE HOSPITALIER BOURG BRESSE FLEYRIAT,ROUTE PARIS,01451,01
2,581,010000032,26010003700068,CENTRE HOSPITALIER BUGEY SUD,AVENUE NARVIK,01034,01
5,584,010000065,26010028400017,CENTRE HOSPITALIER TREVOUX MONTPENSIER,RUE HOPITAL,01427,01
7,586,010000081,26010010200011,CENTRE HOSPITALIER PAYS GEX,RUE MARC PANISSOD,01173,01
8,587,010000099,26010013600019,CENTRE HOSPITALIER MEXIMIEUX,AVENUE DOCTEUR BOYER,01244,01


## 3. Phase 0 — validation des codes commune

Le code commune n'est pas seulement une donnée à contrôler : il porte le
blocking. Il détermine le département, donc la maille communale de la phase 2 et
la maille départementale de la phase 3. Un code obsolète compare une structure
aux candidats d'une commune qui n'existe plus, et elle ressort sans candidat.

Chaque code est confronté à la table de passage INSEE.

| Statut | Situation | Action |
|---|---|---|
| VALIDE | code actif au 01/01/2026 | conservé |
| ARRONDISSEMENT | arrondissement de Paris, Lyon, Marseille | conservé |
| CORRIGE | code disparu, successeur unique | remplacé |
| AMBIGU | code disparu, plusieurs successeurs | conservé, successeurs listés |
| HORS_COG | préfixe 97/98 absent de la table | conservé |
| INCONNU | absent sans explication | conservé, signalé |
| ABSENT | code vide | conservé |

Un code toujours actif est conservé même après une fusion ou une scission : sur
les 23 codes scindés de la table, 22 survivent. Aucune structure n'est écartée ;
les cas douteux portent `cog_anomalie` et la liste de leurs successeurs.

In [4]:
table_cog = cog.charger_table_passage(COG_TABLE_PASSAGE)
index_cog = cog.construire_index(table_cog)

Table de passage : 36,743 correspondances
   codes initiaux distincts : 36,714
   communes actives en 2026 : 34,875
   codes disparus : 1,839, dont 1,838 à successeur unique


In [5]:
df_pm = cog.valider(df_pm, "TX_CogCommune", index_cog)
df_ege = cog.valider(df_ege, "TX_CogCommune", index_cog)

rapport_pm = cog.rapport(df_pm, "PM")
rapport_ege = cog.rapport(df_ege, "EGE")

afficher_tableau(rapport_pm, "Codes commune — personnes morales", max_lignes=10)
afficher_tableau(rapport_ege, "Codes commune — etablissements", max_lignes=10)

Validation COG sur 56,193 PM
   codes modifiés            : 5
   dont changement de dept.  : 0
   anomalies signalées       : 0
Validation COG sur 104,649 EGE
   codes modifiés            : 7
   dont changement de dept.  : 0
   anomalies signalées       : 1


Statut,Nombre,Part
VALIDE,52301,93.07 %
ARRONDISSEMENT,3837,6.83 %
CORRIGE,5,0.01 %
HORS_COG,50,0.09 %
TOTAL,56193,100 %


Statut,Nombre,Part
VALIDE,98624,94.24 %
ARRONDISSEMENT,5929,5.67 %
CORRIGE,7,0.01 %
HORS_COG,88,0.08 %
ABSENT,1,0.00 %
TOTAL,104649,100 %


In [6]:
df_pm = cog.appliquer(df_pm, "cdcommune_norm_pm", "dept_pm")
df_ege = cog.appliquer(df_ege, "cdcommune_norm_ege", "dept_ege")

df_pm.to_parquet(FINESS_PM_CLEAN, index=False)
df_ege.to_parquet(FINESS_EGE_CLEAN, index=False)
print(f"\nParquets reecrits avec les codes valides.")


Parquets reecrits avec les codes valides.


In [7]:
DOSSIER_COG = RESULTS_DIR / "phase0_cog"
DOSSIER_COG.mkdir(parents=True, exist_ok=True)
FICHIER_COG = DOSSIER_COG / "phase0_codes_commune.xlsx"

COLS_PM = ["ID_PmSmsse", "NB_PmSmsseId", "TX_NumFinessPm", "TX_DenominationPm",
           "cog_code_origine", "cog_code_retenu", "cog_statut", "cog_successeurs",
           "cog_libelle", "cog_anomalie", "cdcommune_norm_pm", "dept_pm"]
COLS_EGE = ["ID_Ege", "NB_EgeId", "NB_PmSmsseId", "TX_NumFinessEge", "TX_NomEgeLong",
            "cog_code_origine", "cog_code_retenu", "cog_statut", "cog_successeurs",
            "cog_libelle", "cog_anomalie", "cdcommune_norm_ege", "dept_ege"]

with pd.ExcelWriter(FICHIER_COG, engine="openpyxl") as writer:
    rapport_pm.to_excel(writer, sheet_name="Synthese_PM", index=False)
    rapport_ege.to_excel(writer, sheet_name="Synthese_EGE", index=False)

    for nom, df_src, cols in (("PM", df_pm, COLS_PM), ("EGE", df_ege, COLS_EGE)):
        cols = [c for c in cols if c in df_src.columns]
        corriges = df_src[df_src["cog_statut"] == cog.CORRIGE]
        if len(corriges):
            corriges[cols].to_excel(writer, sheet_name=f"{nom}_corriges", index=False)
        anomalies = df_src[df_src["cog_anomalie"]]
        if len(anomalies):
            anomalies[cols].to_excel(writer, sheet_name=f"{nom}_a_verifier", index=False)

print(f"Rapport : {FICHIER_COG}")

Rapport : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/results/phase0_cog/phase0_codes_commune.xlsx


## 4. Etablissements SIRENE

In [8]:
def traiter_par_lots(source, destination, fonction, taille=TAILLE_CHUNK_PARQUET):
    """Pretraite un gros parquet par lots, sans jamais tout charger.

    Le schema est fige sur le premier lot, colonnes entierement nulles typees en
    string : sans cela, un lot ou une colonne ne contient que des valeurs nulles
    se voit attribuer le type null par pyarrow et l'ecriture echoue.
    """
    writer, schema, total = None, None, 0
    for i, lot in enumerate(pq.ParquetFile(source).iter_batches(batch_size=taille)):
        propre = fonction(lot.to_pandas())
        if schema is None:
            brut = pa.Table.from_pandas(propre, preserve_index=False).schema
            schema = pa.schema([
                pa.field(f.name, pa.string()) if pa.types.is_null(f.type) else f
                for f in brut
            ])
            writer = pq.ParquetWriter(destination, schema)
        writer.write_table(pa.Table.from_pandas(propre, schema=schema,
                                                preserve_index=False))
        total += len(propre)
        print(f"   lot {i + 1} — {total:,} lignes", end="\r")
    if writer:
        writer.close()
    print(f"\n   {total:,} lignes -> {destination}")
    return total

traiter_par_lots(SIRENE_ETAB_RAW, SIRENE_ETAB_CLEAN, pretraiter_etab)


   lot 34 — 16,796,675 lignes
   16,796,675 lignes -> /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/processed/sirene_etab_clean.parquet


16796675

## 5. Unites legales SIRENE

In [9]:
traiter_par_lots(SIRENE_UL_RAW, SIRENE_UL_CLEAN, pretraiter_ul)

   lot 31 — 15,111,205 lignes
   15,111,205 lignes -> /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/processed/sirene_ul_clean.parquet


15111205

## 6. Partitionnement par departement

Operation unique, amortie des la premiere execution des phases 2. On ecrit
departement par departement pour ne pas charger la table entiere.

In [10]:
def _schema_reference(source, colonne_dept, taille):
    """Schema stable, colonnes entierement nulles typees en string."""
    lot = next(pq.ParquetFile(source).iter_batches(batch_size=taille))
    df = lot.to_pandas().drop(columns=[colonne_dept], errors="ignore")
    schema = pa.Table.from_pandas(df, preserve_index=False).schema
    return pa.schema([
        pa.field(f.name, pa.string()) if pa.types.is_null(f.type) else f
        for f in schema
    ])


def partitionner(source, destination, colonne_dept, taille=TAILLE_CHUNK_PARQUET):
    """Ecrit un parquet partitionne par departement, en streaming.

    Chaque lot est force sur un schema de reference : une colonne entierement
    nulle dans un lot prendrait sinon le type null et l'ecriture echouerait sur
    un fichier deja ouvert avec un autre type.
    """
    destination = Path(destination)
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)

    schema = _schema_reference(source, colonne_dept, taille)
    total, writers = 0, {}

    for lot in pq.ParquetFile(source).iter_batches(batch_size=taille):
        df = lot.to_pandas()
        for dept, sous in df.groupby(colonne_dept, sort=False):
            dossier = destination / f"{colonne_dept}={dept}"
            dossier.mkdir(parents=True, exist_ok=True)
            table = pa.Table.from_pandas(sous.drop(columns=[colonne_dept]),
                                         schema=schema, preserve_index=False)
            if dept not in writers:
                writers[dept] = pq.ParquetWriter(dossier / "part.parquet", schema)
            writers[dept].write_table(table)
        total += len(df)
        print(f"   {total:,} lignes, {len(writers)} departements", end="\r")

    for w in writers.values():
        w.close()
    print(f"\n   {total:,} lignes sur {len(writers)} departements -> {destination}")

partitionner(SIRENE_UL_CLEAN, SIRENE_UL_PARTITIONNE, "dept_ul")


   15,111,205 lignes, 110 departements
   15,111,205 lignes sur 110 departements -> /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/processed/sirene_ul_clean_par_dept


In [11]:
partitionner(SIRENE_ETAB_CLEAN, SIRENE_ETAB_PARTITIONNE, "dept_etab")

   16,796,675 lignes, 111 departements
   16,796,675 lignes sur 111 departements -> /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/processed/sirene_etab_clean_par_dept


In [12]:
# Verification : la lecture d'un departement donne-t-elle le contenu attendu ?
dept_test = "75"
bloc = pd.read_parquet(SIRENE_UL_PARTITIONNE, filters=[("dept_ul", "==", dept_test)])
print(f"Departement {dept_test} : {len(bloc):,} unites legales")
print(f"Memoire : {bloc.memory_usage(deep=True).sum() / 1e9:.2f} Go")
afficher_tableau(bloc.head(), f"UL du departement {dept_test}")

Departement 75 : 1,231,198 unites legales
Memoire : 1.19 Go


siren,denominationUniteLegale,sigleUniteLegale,categorieJuridiqueUniteLegale,activitePrincipaleUniteLegale,dateCreationUniteLegale,nicSiegeUniteLegale,numeroVoieEtablissement,typeVoieEtablissement,libelleVoieEtablissement,codeCommuneEtablissement,denomination_norm_ul,sigle_norm_ul,numero_voie_norm_ul,type_voie_norm_ul,libelle_voie_norm_ul,code_commune_norm_ul,libelle_voie_complet_ul,adresse_siege_complete_ul,dept_ul
005520325,ETABLISSEMENTS ADRIEN RIQUIER,None,5710,46.74B,1955-01-01,00027,36,AVENUE,JUNOT,75118,ETABLISSEMENTS ADRIEN RIQUIER,,36,AVENUE,JUNOT,75118,AVENUE JUNOT,36 AVENUE JUNOT,75
005580501,SGN PARIS INVESTISSEMENTS,S.G.N.,5710,66.30Z,1900-01-01,00046,93,BOULEVARD,MALESHERBES,75108,SGN PARIS INVESTISSEMENTS,S G N,93,BOULEVARD,MALESHERBES,75108,BOULEVARD MALESHERBES,93 BOULEVARD MALESHERBES,75
005780580,IMMOBILIERE LOUIS THIRIET,ILT,5499,64.30Z,1957-01-01,00030,52,RUE,DE LA BIENFAISANCE,75108,IMMOBILIERE LOUIS THIRIET,ILT,52,RUE,BIENFAISANCE,75108,RUE BIENFAISANCE,52 RUE DE LA BIENFAISANCE,75
006580195,SOCIETE INDUSTRIELLE POUR LE DEVELOPPEMENT DE LA SECURITE,SIDES,5710,29.10Z,1965-01-01,00078,32,AVENUE,KLEBER,75116,SOCIETE INDUSTRIELLE DEVELOPPEMENT SECURITE,SIDES,32,AVENUE,KLEBER,75116,AVENUE KLEBER,32 AVENUE KLEBER,75
007071806,[ND],[ND],1000,68.20A,1981-03-17,00041,[ND],[ND],[ND],75110,ND,ND,,ND,ND,75110,ND ND,[ND] [ND] [ND],75
